# Bark (Small) — Toy Multi-Stage GPT for Text-to-Audio

In [ ]:
import mathimport torchimport torch.nn as nnimport torch.nn.functional as Ffrom torch.utils.data import Dataset, DataLoader

In [ ]:
class Config:    text_vocab_size = 256    semantic_vocab_size = 512    coarse_vocab_size = 512    fine_vocab_size = 512    n_fine_codebooks = 4    d_model = 128    n_heads = 4    n_layers = 2    block_size = 128    dropout = 0.1cfg = Config()device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
class CausalSelfAttention(nn.Module):    def __init__(self, cfg):        super().__init__()        self.n_heads = cfg.n_heads        self.head_dim = cfg.d_model // cfg.n_heads        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model)        self.proj = nn.Linear(cfg.d_model, cfg.d_model)        self.dropout = nn.Dropout(cfg.dropout)        mask = torch.tril(torch.ones(cfg.block_size, cfg.block_size))        self.register_buffer("mask", mask.unsqueeze(0).unsqueeze(0))    def forward(self, x):        b, t, c = x.shape        qkv = self.qkv(x).view(b, t, 3, self.n_heads, self.head_dim).permute(2, 0, 3, 1, 4)        q, k, v = qkv[0], qkv[1], qkv[2]        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)        att = att.masked_fill(self.mask[:, :, :t, :t] == 0, float("-inf"))        att = F.softmax(att, dim=-1)        att = self.dropout(att)        y = att @ v        y = y.transpose(1, 2).contiguous().view(b, t, c)        return self.proj(y)

In [ ]:
class MLP(nn.Module):    def __init__(self, cfg):        super().__init__()        self.fc1 = nn.Linear(cfg.d_model, 4 * cfg.d_model)        self.fc2 = nn.Linear(4 * cfg.d_model, cfg.d_model)        self.dropout = nn.Dropout(cfg.dropout)    def forward(self, x):        x = F.gelu(self.fc1(x))        x = self.fc2(x)        return self.dropout(x)

In [ ]:
class Block(nn.Module):    def __init__(self, cfg):        super().__init__()        self.ln1 = nn.LayerNorm(cfg.d_model)        self.attn = CausalSelfAttention(cfg)        self.ln2 = nn.LayerNorm(cfg.d_model)        self.mlp = MLP(cfg)    def forward(self, x):        x = x + self.attn(self.ln1(x))        x = x + self.mlp(self.ln2(x))        return x

In [ ]:
class GPTStage(nn.Module):    def __init__(self, cfg, in_vocab_size, out_vocab_size, cond_dim=None):        super().__init__()        self.tok_emb = nn.Embedding(in_vocab_size, cfg.d_model)        self.pos_emb = nn.Parameter(torch.zeros(1, cfg.block_size, cfg.d_model))        self.cond_proj = nn.Linear(cond_dim, cfg.d_model) if cond_dim else None        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])        self.ln_f = nn.LayerNorm(cfg.d_model)        self.head = nn.Linear(cfg.d_model, out_vocab_size)    def forward(self, idx, cond=None):        b, t = idx.shape        x = self.tok_emb(idx) + self.pos_emb[:, :t]        if cond is not None and self.cond_proj is not None:            x = x + self.cond_proj(cond).unsqueeze(1)        for block in self.blocks:            x = block(x)        x = self.ln_f(x)        return self.head(x)

In [ ]:
class FineStage(nn.Module):    def __init__(self, cfg):        super().__init__()        self.codebook_embs = nn.ModuleList([            nn.Embedding(cfg.fine_vocab_size, cfg.d_model) for _ in range(cfg.n_fine_codebooks)        ])        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])        self.ln_f = nn.LayerNorm(cfg.d_model)        self.heads = nn.ModuleList([            nn.Linear(cfg.d_model, cfg.fine_vocab_size) for _ in range(cfg.n_fine_codebooks)        ])    def forward(self, coarse_tokens):        x = sum(emb(coarse_tokens) for emb in self.codebook_embs) / len(self.codebook_embs)        for block in self.blocks:            x = block(x)        x = self.ln_f(x)        return [head(x) for head in self.heads]

In [ ]:
class BarkModel(nn.Module):    def __init__(self, cfg):        super().__init__()        self.semantic_model = GPTStage(cfg, cfg.text_vocab_size, cfg.semantic_vocab_size)        self.coarse_model = GPTStage(cfg, cfg.semantic_vocab_size, cfg.coarse_vocab_size, cond_dim=cfg.d_model)        self.fine_model = FineStage(cfg)        self.semantic_summary = nn.Embedding(cfg.semantic_vocab_size, cfg.d_model)    def forward(self, text_tokens, semantic_tokens, coarse_tokens):        semantic_logits = self.semantic_model(text_tokens)        semantic_cond = self.semantic_summary(semantic_tokens).mean(dim=1)        coarse_logits = self.coarse_model(coarse_tokens, cond=semantic_cond)        fine_logits = self.fine_model(coarse_tokens)        return semantic_logits, coarse_logits, fine_logits

In [ ]:
class ToyBarkDataset(Dataset):    def __init__(self, n_samples, cfg):        self.n_samples = n_samples        self.cfg = cfg    def __len__(self):        return self.n_samples    def __getitem__(self, idx):        seq_len = self.cfg.block_size        text_tokens = torch.randint(1, self.cfg.text_vocab_size, (seq_len,))        semantic_tokens = torch.randint(1, self.cfg.semantic_vocab_size, (seq_len,))        coarse_tokens = torch.randint(1, self.cfg.coarse_vocab_size, (seq_len,))        fine_tokens = torch.randint(1, self.cfg.fine_vocab_size, (self.cfg.n_fine_codebooks, seq_len))        return text_tokens, semantic_tokens, coarse_tokens, fine_tokens

In [ ]:
dataset = ToyBarkDataset(n_samples=32, cfg=cfg)loader = DataLoader(dataset, batch_size=4, shuffle=True)model = BarkModel(cfg).to(device)optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

In [ ]:
def train_epoch(model, loader, optimizer):    model.train()    total_loss = 0.0    for text_tokens, semantic_tokens, coarse_tokens, fine_tokens in loader:        text_tokens = text_tokens.to(device)        semantic_tokens = semantic_tokens.to(device)        coarse_tokens = coarse_tokens.to(device)        fine_tokens = fine_tokens.to(device)        semantic_logits, coarse_logits, fine_logits = model(text_tokens, semantic_tokens, coarse_tokens)        semantic_loss = F.cross_entropy(semantic_logits.reshape(-1, cfg.semantic_vocab_size), semantic_tokens.reshape(-1))        coarse_loss = F.cross_entropy(coarse_logits.reshape(-1, cfg.coarse_vocab_size), coarse_tokens.reshape(-1))        fine_loss = sum(            F.cross_entropy(fine_logits[i].reshape(-1, cfg.fine_vocab_size), fine_tokens[:, i, :].reshape(-1))            for i in range(cfg.n_fine_codebooks)        ) / cfg.n_fine_codebooks        loss = semantic_loss + coarse_loss + fine_loss        optimizer.zero_grad()        loss.backward()        optimizer.step()        total_loss += loss.item()    return total_loss / len(loader)

In [ ]:
for epoch in range(5):    avg_loss = train_epoch(model, loader, optimizer)    print(f"epoch {epoch + 1} loss {avg_loss:.4f}")

In [ ]:
model.eval()with torch.no_grad():    text_prompt = torch.randint(1, cfg.text_vocab_size, (1, cfg.block_size)).to(device)    semantic_logits = model.semantic_model(text_prompt)    semantic_tokens = semantic_logits.argmax(dim=-1)    semantic_cond = model.semantic_summary(semantic_tokens).mean(dim=1)    coarse_logits = model.coarse_model(semantic_tokens, cond=semantic_cond)    coarse_tokens = coarse_logits.argmax(dim=-1)    fine_logits = model.fine_model(coarse_tokens)    fine_tokens = torch.stack([f.argmax(dim=-1) for f in fine_logits], dim=1)    print(semantic_tokens.shape, coarse_tokens.shape, fine_tokens.shape)